**Author:** Salvador Navas  
**Date:** 2025-06-27

### GPM 3IMERG — Multi-satellite Precipitation (NASA)

**GPM IMERG** (Integrated Multi-satellitE Retrievals for GPM) is a high-resolution global
precipitation product from NASA combining passive microwave sensors, the GPM radar, and
surface gauge data.

| Property | Value |
|----------|-------|
| Spatial resolution | ~0.1° (~10 km) |
| Temporal resolution | 30 min (half-hourly), hourly, daily, monthly |
| Coverage | 60°S – 60°N |
| Period | June 2000 – present (TRMM-era back-extension from 1998) |
| Access | NASA Earthdata (free account) |

#### 🔄 Three processing runs

| Run | Latency | Gauge correction | Use case |
|-----|---------|-----------------|----------|
| **Early** | ~4 hours | None | Near-real-time flood monitoring |
| **Late** | ~14 hours | None | Operational forecasting |
| **Final (3IMERG)** | ~3 months | GPCC gauges | ✅ Research, hydrological modelling |

> Always use the **Final Run** for historical analysis — it is the only version with gauge correction.

#### 🆚 GPM IMERG vs PERSIANN-CCS

| Feature | GPM IMERG Final | PERSIANN-CCS |
|---------|----------------|--------------|
| Algorithm | Multi-sensor (MW + IR + gauge) | IR-only + ANN |
| Accuracy (mid-latitudes) | **Better** | Lower |
| Accuracy (tropics) | Better | Moderate |
| Temporal resolution | 30 min | 1 hour |
| Spatial resolution | ~10 km | **~4 km** |
| Near real-time latency | 4 hours (Early) | ~1 day |
| Period | 2000–now | 2003–now |

> **Use GPM IMERG** for historical analysis, bias correction, or when accuracy matters.
> **Use PERSIANN-CCS** when you need finer spatial detail (~4 km) or faster updates.

#### 🔑 NASA Earthdata credentials setup

1. Register at https://urs.earthdata.nasa.gov (free)
2. Accept the GESDISC data access agreement in your Earthdata profile
3. Set credentials via environment variables:
   ```bash
   export EARTHDATA_USERNAME=<your-username>
   export EARTHDATA_PASSWORD=<your-password>
   ```
   — or add to `~/.netrc`:
   ```
   machine urs.earthdata.nasa.gov login <username> password <password>
   ```

#### 🔗 Technical reference
- Huffman et al. (2019). NASA GPM IMERG ATBD v06. https://doi.org/10.5067/GPM/IMERG/3B-HH/06


In [1]:
from pyhydra.data_sources.rainfall import GPMDownloader
from pyhydra.utils import interactive_map
import os
from pathlib import Path
RUN_DOWNLOADS = os.getenv('HYDRA_RUN_DOWNLOADS', '0') == '1'
OUTPUT_DIR = Path(os.getenv('HYDRA_RUNTIME_DIR', str(Path.cwd() / '.hydra_runtime'))) / 'gpm'
print(f'Run downloads: {RUN_DOWNLOADS} | Output dir: {OUTPUT_DIR}')


Run downloads: False | Output dir: /Users/salvadornavasfernandez/Desktop/Github/HYDRA/notebooks/data_sources/rainfall/.hydra_runtime/gpm


## 🗺️ Select area / point of interest

Draw a **rectangle** for a gridded spatial download, or click a **point** for a single-location time series.

| Mode | When to use | Output |
|------|-------------|--------|
| **Point** | Time series at a gauge or city | `pandas.Series` |
| **Rectangle** | Spatial maps, catchment-average | NetCDF files |

> Gridded downloads can be large: 1 month of half-hourly at 0.1° over Iberia (~500×400 km²)
> ≈ **300 MB**. Use `resolution='daily'` for water balance studies.


In [2]:
coordinates_list = interactive_map(zoom=4, center=(20, 0))

---
## ⚙️ Download configuration

```python
gpm = GPMDownloader()
gpm.set_region(
    points=[(lat, lon)],          # point mode
    # OR:
    bbox=(lon_min, lat_min, lon_max, lat_max),  # gridded mode
)
results = gpm.search(
    start_date='2024-10-01',
    end_date='2024-10-31',
    resolution='hourly',          # 'half_hourly', 'hourly', 'daily', 'monthly'
)
data = gpm.open_dataset(results, variable='precipitation',
                        output_folder='/workspace/data/gpm/')
```

#### Choosing resolution

| Resolution | File size (1 month, Iberia) | Use case |
|------------|---------------------------|----------|
| Half-hourly | ~600 MB | Flash flood analysis, storm tracking |
| **Hourly** | ~300 MB | Urban drainage, event hyetographs |
| **Daily** | ~12 MB | Water balance, bias correction |
| Monthly | ~0.4 MB | Climatology, trend analysis |

#### Variable units

- `precipitation`: mm/hr (precipitation rate)
- To convert to daily total: `sum(hourly_values)` = mm/day (since each value is mm/hr × 1hr)


In [3]:
if RUN_DOWNLOADS:
    # 1. Create the downloader and authenticate (NASA Earthdata credentials required)
    #    First time: earthaccess will prompt for your username and password.
    #    Register at https://urs.earthdata.nasa.gov/
    gpm = GPMDownloader()

    # 2. Define the region of interest
    # Option A — from the interactive map above:
    # points = [coordinates_list[0]['value']]  # for a point selection
    # area   = coordinates_list[0]             # for a rectangle selection

    # Option B — hardcode coordinates:
    points = [(39.392231, -0.624717)]   # Valencia
    gpm.set_region(points=points)

    # 3. Search and download
    results = gpm.search(
        start_date='2024-10-01',
        end_date='2024-10-31',
        resolution='hourly'
    )
    data = gpm.open_dataset(results, variable='precipitation', output_folder=str(OUTPUT_DIR))
else:
    print('GPM download skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 in an admin/session environment to run it.')
    data = None

GPM download skipped in public mode. Set HYDRA_RUN_DOWNLOADS=1 in an admin/session environment to run it.


In [4]:
if data is not None:
    data.plot()
else:
    print("Gridded mode: data saved as NetCDF files in the output folder. Use xr.open_dataset() to load them.")

Gridded mode: data saved as NetCDF files in the output folder. Use xr.open_dataset() to load them.


---
## 📊 Using GPM data in hydrological workflows

#### Catchment-average precipitation

```python
import xarray as xr
ds = xr.open_mfdataset('/workspace/data/gpm/*.nc4', combine='by_coords')
# Simple spatial mean over catchment bounding box
catchment_mean = ds['precipitation'].mean(dim=['lat', 'lon'])
daily_mm = catchment_mean.resample(time='1D').sum()  # hourly sum → mm/day
```

#### Bias correction against gauges

GPM IMERG Final is generally well-calibrated but regional biases exist:
- **Mediterranean Spain**: slight underestimation during DANA events (deep convection)
- **Pyrenees / Cantabrian**: underestimation of orographic enhancement

```python
# Monthly multiplicative correction
bias_factor = aemet_monthly_mean / gpm_monthly_mean
gpm_corrected = gpm_daily * bias_factor
```

#### ⚠️ Known limitations

- No data poleward of 60°N/S (passive microwave sensors)
- Solid precipitation (snow) detection is less accurate
- The Final Run has ~3 month latency — use Early/Late for operational applications
- 0.1° resolution (~10 km) may miss isolated convective cells
